In [2]:
import re
import pandas as pd

CODEBOOK = "Codebook_HSLS_17_PETS.txt"

In [3]:
text = open(CODEBOOK, encoding="utf-8-sig").read()
entries = re.split(r"-{80,}", text)
print(f"Number of blocks: {len(entries)}")

Number of blocks: 20603


In [4]:
# Find a block containing a variable you know well
for i, entry in enumerate(entries):
    if "Name:       X1SES" in entry:
        print(f"Block index: {i}")
        print(entry[:1500])
        break

Block index: 210



File:       STUDENT
Name:       X1SES
Position:   587
Length:     7
Label:      X1 Socio-economic status composite

Description:
This composite variable is used to measure a construct for socioeconomic status. X1SES is calculated using parent/guardians' education (X1PAR1EDU and X1PAR2EDU), occupation (X1PAR1OCC2 and X1PAR2OCC2), and family income (X1FAMINCOME). For cases with nonresponding parent/guardians, 5 imputed values are generated (X1SES1-X1SES5), X1SES is computed as the average of the 5 imputed values, and the imputation flag is set as X1SES_IM=1 (values for parent/guardian education, occupation, and income are set to -8). When education, occupation, or family income are imputed using other information provided by the responding parent/guardian, X1SES is constructed from the combination of actual and imputed parent/guardian values. For these cases, the values of X1SES1-X1SES5 are equivalent to X1SES and X1SES_IM=2. Otherwise, the responding parent/guardian 

In [5]:
for i, entry in enumerate(entries):
    if "Name:       X3DROPSTAT" in entry:
        print(entry[:2000])
        break




File:       STUDENT
Name:       X3DROPSTAT
Position:   1352
Length:     2
Label:      X3 U13 dropout status

Description:
Sample member status of dropout from high school.


SAS Logic:
if X3SQSTAT=8 then X3DROPSTAT=-8;
else if X3HSCRED=1 and X3HSCREDTYPE=1 then X3DROPSTAT=0;
else if X3HSCRED=0 and ((201300<X3LASTHSDATE<201304) or 0<X3LASTHSDATE<201300) then X3DROPSTAT=1;
else if X3HSCRED=1 and X3HSCREDTYPE in (2,3) then X3DROPSTAT=2;
else X3DROPSTAT=4;
if X3DROPSTAT in (0,4) and X2EVERDROP=1 then X3DROPSTAT=3;
Sources: 2013 update questionnaire

                                                                        Frequency             Percent 
Category            Label                                              Unweighted          Unweighted 
------------------- ----------------------------------------- ------------------- ------------------- 
0                   Not dropout/alternative completer                      15,535               66.10 
1                   Spring term 2

In [6]:
def parse_entry(entry):
    name_m    = re.search(r"Name:\s+(\S+)", entry)
    label_m   = re.search(r"Label:\s+(.+)", entry)
    desc_m    = re.search(r"Description:\s*\n(.*?)(?=SAS Logic:|Sources:|Category\s+Label|Mean\s+Std|\Z)", entry, re.DOTALL)
    sas_m     = re.search(r"SAS Logic:\s*\n(.*?)(?=Sources:|Category\s+Label|Mean\s+Std|\Z)", entry, re.DOTALL)
    src_m     = re.search(r"Sources?:\s*(.+?)(?:\n\n|\Z)", entry, re.DOTALL)

    if not name_m:
        return None

    return {
        "variable":    name_m.group(1).strip(),
        "label":       label_m.group(1).strip() if label_m else "",
        "description": desc_m.group(1).strip()  if desc_m  else "",
        "sas_logic":   sas_m.group(1).strip()   if sas_m   else "",
        "sources":     src_m.group(1).strip()    if src_m   else "",
    }

parsed = [parse_entry(e) for e in entries]
parsed = [p for p in parsed if p is not None]
print(f"Successfully parsed: {len(parsed)}")

Successfully parsed: 10301


In [7]:
# Check a description composite
x1ses = next(p for p in parsed if p["variable"] == "X1SES")
print("=== X1SES ===")
print(f"Label:       {x1ses['label']}")
print(f"Description: {x1ses['description'][:200]}")
print(f"SAS Logic:   '{x1ses['sas_logic']}'")
print(f"Sources:     {x1ses['sources']}")

print()

# Check a SAS composite
x3drop = next(p for p in parsed if p["variable"] == "X3DROPSTAT")
print("=== X3DROPSTAT ===")
print(f"Label:       {x3drop['label']}")
print(f"Description: {x3drop['description'][:200]}")
print(f"SAS Logic:   {x3drop['sas_logic'][:200]}")
print(f"Sources:     {x3drop['sources']}")

=== X1SES ===
Label:       X1 Socio-economic status composite
Description: This composite variable is used to measure a construct for socioeconomic status. X1SES is calculated using parent/guardians' education (X1PAR1EDU and X1PAR2EDU), occupation (X1PAR1OCC2 and X1PAR2OCC2)
SAS Logic:   ''
Sources:     

=== X3DROPSTAT ===
Label:       X3 U13 dropout status
Description: Sample member status of dropout from high school.
SAS Logic:   if X3SQSTAT=8 then X3DROPSTAT=-8;
else if X3HSCRED=1 and X3HSCREDTYPE=1 then X3DROPSTAT=0;
else if X3HSCRED=0 and ((201300<X3LASTHSDATE<201304) or 0<X3LASTHSDATE<201300) then X3DROPSTAT=1;
else if X3H
Sources:     2013 update questionnaire


In [8]:
col_names = open("hsls_column_names.txt").read().strip().split("\n")

parsed_names = {p["variable"] for p in parsed}
missing = [v for v in col_names if v not in parsed_names]

print(f"Your variables:     {len(col_names)}")
print(f"Found in codebook:  {len(col_names) - len(missing)}")
print(f"Missing:            {len(missing)}")
print(missing)

Your variables:     913
Found in codebook:  913
Missing:            0
[]


In [9]:
codebook_df = pd.DataFrame(parsed)
codebook_df = codebook_df[codebook_df["variable"].isin(col_names)].reset_index(drop=True)
print(codebook_df.shape)
codebook_df.head()

(1125, 5)


,variable,label,description,sas_logic,sources
0,STU_ID,Student ID,Student identifier assigned for all base year ...,,
1,X1SEX,X1 Student's sex,"Sex of the sample member, taken from the base ...",,
2,X1RACE,X1 Student's race/ethnicity-composite,X1RACE characterizes the sample member's race/...,,
3,X1HISPANIC,X1 Student is Hispanic/Latino/Latina-composite,The sample member's race/ethnicity is characte...,,
4,X1WHITE,X1 Student is White-composite,The sample member's race/ethnicity is characte...,,


In [10]:
dups = codebook_df[codebook_df.duplicated(subset="variable", keep=False)]
print(f"Duplicate variable names: {dups['variable'].nunique()}")
print(dups["variable"].value_counts().head(20))

Duplicate variable names: 212
variable
X1CONTROL       2
X1LOCALE        2
X1REGION        2
X1SCHOOLCLI     2
X1COUPERTEA     2
X1COUPERCOU     2
X1COUPERPRI     2
X1AQSTAT        2
X1AQDATE        2
X1AQDESIGNEE    2
X1CQSTAT        2
X1CQDATE        2
A1SCHCONTROL    2
A1NOTIFY        2
A1MTHSCIFAIR    2
A1MSSUMMER      2
A1MSAFTERSCH    2
A1MSMENTOR      2
A1MSSPEAKER     2
A1MSFLDTRIP     2
Name: count, dtype: int64


In [11]:
codebook_df = codebook_df.drop_duplicates(subset="variable", keep="first").reset_index(drop=True)
print(codebook_df.shape)

(913, 5)


In [12]:
def get_instrument(v):
    if v.startswith("X1"):   return "NCES_composite"
    if v.startswith("S1"):   return "student_BY"
    if v.startswith("P1"):   return "parent_BY"
    if v.startswith("M1"):   return "mathteacher_BY"
    if v.startswith("A1"):   return "admin_BY"
    if v.startswith("C1"):   return "counselor_BY"
    if v == "STU_ID":        return "identifier"
    if v == "X4EVERDROP":    return "target"
    return "other"

codebook_df["instrument"] = codebook_df["variable"].apply(get_instrument)
codebook_df["instrument"].value_counts()

instrument
student_BY        271
NCES_composite    161
counselor_BY      150
parent_BY         141
mathteacher_BY    134
admin_BY           54
identifier          1
target              1
Name: count, dtype: int64

In [13]:
def get_construct_type(row):
    if row["sas_logic"]:
        return "sas_composite"
    if any(kw in row["description"].lower() for kw in [
        "calculated using", "inputs to", "constructed from",
        "derived from", "based on", "computed from",
        "created from", "uses ", "using "
    ]):
        return "description_composite"
    return "raw"

codebook_df["construct_type"] = codebook_df.apply(get_construct_type, axis=1)
codebook_df["construct_type"].value_counts()

construct_type
raw                      704
description_composite    208
sas_composite              1
Name: count, dtype: int64

In [14]:
# Tokens to ignore when scanning for variable name references
IGNORE_TOKENS = {
    "IF", "THEN", "ELSE", "AND", "OR", "NOT", "IN", "DO", "END",
    "NULL", "MISSING", "MAX", "MIN", "SUM", "MEAN", "ABS", "INT",
    "SUBSTR", "UPCASE", "PUT", "INPUT", "CAT", "CATS", "CATX",
    "LENGTH", "LABEL", "FORMAT", "INFORMAT", "RETAIN", "ARRAY",
    "OUTPUT", "RETURN", "STOP", "RUN", "DATA", "SET", "MERGE",
    "BY", "WHERE", "KEEP", "DROP", "RENAME", "CLASS", "MODEL",
    "VAR", "WEIGHT", "FREQ", "TABLES", "TRUE", "FALSE",
    "NCESID", "CCD", "PSS", "NCES", "GED", "IEP", "AP", "IB",
    "ONET", "STEM", "SAT", "ACT", "PSAT", "GPA",
}

VARNAME_RE = re.compile(r'\b([A-Z][A-Z0-9_]{2,})\b')
all_known  = set(codebook_df["variable"])

def extract_parents(row):
    text = row["sas_logic"] if row["sas_logic"] else row["description"]
    candidates = VARNAME_RE.findall(text)
    parents = set()
    for c in candidates:
        if c == row["variable"]:      continue
        if c in IGNORE_TOKENS:        continue
        if c not in all_known:        continue
        if not any(ch.isdigit() for ch in c): continue
        parents.add(c)
    return sorted(parents)

codebook_df["parents"] = codebook_df.apply(extract_parents, axis=1)
codebook_df["n_parents"] = codebook_df["parents"].apply(len)

# Verify on known variables
for v in ["X1SES", "X1MTHID", "X1SCHOOLBEL", "X3DROPSTAT"]:
    if v in all_known:
        row = codebook_df[codebook_df["variable"] == v].iloc[0]
        print(f"{v}: {row['parents']}")

X1SES: ['X1FAMINCOME', 'X1PAR1EDU', 'X1PAR1OCC2', 'X1PAR2EDU', 'X1PAR2OCC2', 'X1SES1', 'X1SES5', 'X1SES_IM']
X1MTHID: ['S1MPERSON1', 'S1MPERSON2']
X1SCHOOLBEL: ['S1GOODGRADES', 'S1PROUD', 'S1SAFE', 'S1SCHWASTE', 'S1TALKPROB']


In [15]:
def compute_depth_and_leaves(varname, memo, visiting):
    if varname in memo:
        return memo[varname]
    if varname in visiting:
        # cycle detected - break it
        return (0, {varname})
    
    visiting.add(varname)
    
    row = codebook_df[codebook_df["variable"] == varname]
    if row.empty:
        result = (0, {varname})
    else:
        parents = row.iloc[0]["parents"]
        if not parents:
            result = (0, {varname})
        else:
            child_results = [compute_depth_and_leaves(p, memo, visiting) for p in parents]
            max_depth = max(r[0] for r in child_results)
            all_leaves = set().union(*[r[1] for r in child_results])
            result = (max_depth + 1, all_leaves)
    
    visiting.remove(varname)
    memo[varname] = result
    return result

memo = {}
for v in codebook_df["variable"]:
    compute_depth_and_leaves(v, memo, set())

codebook_df["depth"]     = codebook_df["variable"].apply(lambda v: memo[v][0])
codebook_df["n_leaves"]  = codebook_df["variable"].apply(lambda v: len(memo[v][1]))

# Verify
for v in ["X1SES", "X1MTHID", "X1SCHOOLBEL", "S1GOODGRADES"]:
    if v in all_known:
        row = codebook_df[codebook_df["variable"] == v].iloc[0]
        print(f"{v}: depth={row['depth']}, n_leaves={row['n_leaves']}")

X1SES: depth=5, n_leaves=16
X1MTHID: depth=1, n_leaves=2
X1SCHOOLBEL: depth=1, n_leaves=5
S1GOODGRADES: depth=0, n_leaves=1


In [16]:
codebook_df.to_csv("codebook_parsed.csv", index=False)
print(codebook_df.shape)
codebook_df.head()

(913, 11)


,variable,label,description,sas_logic,sources,instrument,construct_type,parents,n_parents,depth,n_leaves
0,STU_ID,Student ID,Student identifier assigned for all base year ...,,,identifier,raw,[],0,0,1
1,X1SEX,X1 Student's sex,"Sex of the sample member, taken from the base ...",,,NCES_composite,description_composite,[],0,0,1
2,X1RACE,X1 Student's race/ethnicity-composite,X1RACE characterizes the sample member's race/...,,,NCES_composite,description_composite,"[X1BLACK, X1HISPANIC, X1WHITE]",3,4,3
3,X1HISPANIC,X1 Student is Hispanic/Latino/Latina-composite,The sample member's race/ethnicity is characte...,,,NCES_composite,description_composite,"[X1BLACK, X1RACE, X1WHITE]",3,2,3
4,X1WHITE,X1 Student is White-composite,The sample member's race/ethnicity is characte...,,,NCES_composite,description_composite,"[X1BLACK, X1HISPANIC, X1RACE]",3,1,3


In [17]:
import json
import os

SAVE_FILE = "tier1_assignments.json"

# Load existing progress if it exists
if os.path.exists(SAVE_FILE):
    assignments = json.load(open(SAVE_FILE))
else:
    assignments = {}

for _, row in codebook_df.iterrows():
    v = row["variable"]
    
    # Skip identifier and target
    if row["instrument"] in ("identifier", "target"):
        continue
    
    # Skip already assigned
    if v in assignments:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable:    {v}")
    print(f"Label:       {row['label']}")
    print(f"Instrument:  {row['instrument']}")
    print(f"Description: {row['description'][:400]}")
    
    tier1 = input("\nTier 1? (y/n/q to quit): ").strip().lower()
    if tier1 == "q":
        break
    
    certain = input("Certain? (y/n): ").strip().lower()
    
    assignments[v] = {
        "tier_1":   tier1 == "y",
        "certain":  certain == "y",
    }
    
    json.dump(assignments, open(SAVE_FILE, "w"), indent=2)

print(f"\nProgress: {len(assignments)}/911 variables assigned")



Progress: 911/911 variables assigned


In [18]:
assignments = json.load(open("tier1_assignments.json"))

for variable, decision in assignments.items():
    if decision["certain"]:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    print(f"Current:  tier_1={decision['tier_1']}")
    
    # Show label from codebook_df
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    tier1 = input("\nTier 1? (y/n/q to quit): ").strip().lower()
    if tier1 == "q":
        break
    
    assignments[variable]["tier_1"] = (tier1 == "y")
    assignments[variable]["certain"] = True
    
    json.dump(assignments, open("tier1_assignments.json", "w"), indent=2)

print("Done")

Done


In [23]:
tier1 = json.load(open("tier1_assignments.json"))
tier1_vars = {k for k, v in tier1.items() if v["tier_1"]}

SAVE_FILE = "tier2_assignments.json"

if os.path.exists(SAVE_FILE):
    assignments = json.load(open(SAVE_FILE))
else:
    assignments = {}

last_variable = None

for _, row in codebook_df.iterrows():
    v = row["variable"]

    if row["instrument"] in ("identifier", "target"):
        continue
    if v in tier1_vars:
        continue
    if v in assignments:
        continue

    print(f"\n{'='*60}")
    print(f"Variable:    {v}")
    print(f"Label:       {row['label']}")
    print(f"Instrument:  {row['instrument']}")
    print(f"Description: {row['description'][:400]}")

    tier2 = input("\nTier 2? (y/n/r to redo last/q to quit): ").strip().lower()
    if tier2 == "q":
        break
    elif tier2 == "r":
        if last_variable:
            assignments[last_variable]["certain"] = False
            json.dump(assignments, open(SAVE_FILE, "w"), indent=2)
            print(f"Marked {last_variable} as uncertain — will reappear next run")
        else:
            print("Nothing to redo")
        continue

    certain = input("Certain? (y/n): ").strip().lower()

    assignments[v] = {
        "tier_2": tier2 == "y",
        "certain": certain == "y",
    }
    last_variable = v

    json.dump(assignments, open(SAVE_FILE, "w"), indent=2)

print(f"\nProgress: {len(assignments)}/{911 - len(tier1_vars)} variables assigned")


Variable:    C1CLGPREP
Label:       C1 B27A School has counselor designated for college readiness/selection/apply
Instrument:  counselor_BY
Description: Does your school have one or more counselors whose primary responsibility is...
assisting students with college readiness, selection, and applications?
  Yes
  No


                                                                        Frequency             Percent

Variable:    C1WORKFORCE
Label:       C1 B27B School has counselor designated for workforce preparation/placement
Instrument:  counselor_BY
Description: Does your school have one or more counselors whose primary responsibility is...
assisting students with preparation for and placement into the workforce?
  Yes
  No


                                                                        Frequency             Percent

Variable:    C1CLGFAIR
Label:       C1 B28A School holds or participates in college fairs
Instrument:  counselor_BY
Description: Which of the following ste

In [26]:
assignments = json.load(open("tier2_assignments.json"))

for variable, decision in assignments.items():
    if decision["certain"]:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    print(f"Current:  tier_2={decision['tier_2']}")
    
    # Show label from codebook_df
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    tier2 = input("\nTier 2? (y/n/q to quit): ").strip().lower()
    if tier2 == "q":
        break
    
    assignments[variable]["tier_2"] = (tier2 == "y")
    assignments[variable]["certain"] = True
    
    json.dump(assignments, open("tier2_assignments.json", "w"), indent=2)

print("Done")


Variable: A1G9COMMUNTY
Current:  tier_2=False
Label:    A1 A26C Offers 9th grade learning communities separate from rest of school
Description: Does your high school offer any of the following programs to assist 9th graders who are struggling academically?
(check all that apply)
  Summer program prior to entry into high school that provides supplemental instruction in reading and math
  Small learning communities or Achievement Academies fo

Variable: A1FILLMTH
Current:  tier_2=False
Label:    A1 C05 Ease of filling high school mathematics teaching vacancies
Description: How easy or difficult was it to fill the high school teaching vacancies in the mathematics department in your school?  Would you say...
  easy
  somewhat difficult
  very difficult or
  you could not fill the vacancies in the math department?


"Could not fill math department" recoded with "Very dif

Variable: A1YRSADMIN
Current:  tier_2=False
Label:    A1 E10 Years served as principal of any school
Description: Inclu

In [38]:
tier1 = json.load(open("tier1_assignments.json"))
tier2 = json.load(open("tier2_assignments.json"))

already_assigned = set()
for k, v in tier1.items():
    if v["tier_1"]:
        already_assigned.add(k)
for k, v in tier2.items():
    if v["tier_2"]:
        already_assigned.add(k)

SAVE_FILE = "exclusions.json"

if os.path.exists(SAVE_FILE):
    exclusions = json.load(open(SAVE_FILE))
else:
    exclusions = {}

last_variable = None

for _, row in codebook_df.iterrows():
    v = row["variable"]

    if row["instrument"] in ("identifier", "target"):
        continue
    if v in already_assigned:
        continue
    if v in exclusions:
        continue

    print(f"\n{'='*60}")
    print(f"Variable:    {v}")
    print(f"Label:       {row['label']}")
    print(f"Instrument:  {row['instrument']}")
    print(f"Description: {row['description'][:400]}")

    exclude = input("\nExclude entirely? (y/n/r to redo last/q to quit): ").strip().lower()
    if exclude == "q":
        break
    elif exclude == "r":
        if last_variable:
            exclusions[last_variable]["certain"] = False
            json.dump(exclusions, open(SAVE_FILE, "w"), indent=2)
            print(f"Marked {last_variable} as uncertain — will reappear next run")
        else:
            print("Nothing to redo")
        continue

    certain = input("Certain? (y/n): ").strip().lower()

    exclusions[v] = {
        "exclude": exclude == "y",
        "certain": certain == "y",
    }
    last_variable = v

    json.dump(exclusions, open(SAVE_FILE, "w"), indent=2)

remaining = 911 - len(already_assigned) - len(exclusions)
print(f"\nProgress: {len(exclusions)} exclusion decisions made, ~{remaining} remaining")


Variable:    C1ENCCLG
Label:       C1 B20C School has program to encourage student not considering college to do so
Instrument:  counselor_BY
Description: Does your school have any formal programs to...
encourage students who might not be considering college to do so?
  Yes
  No


                                                                        Frequency             Percent



Variable:    C1ONLINE
Label:       C1 B21B Courses not offered by school available on-line
Instrument:  counselor_BY
Description: In which of the following ways may a student take a course for credit if it is not offered by your school?
(Check all that apply.)
  Independent study
  On-line or distance learning courses
  Courses at another traditional high school in the district
  Courses at a local career or technical school
  Courses at a local community college
  Courses at a nearby 4-year college or university
  Students 

Variable:    C1OTHERHS
Label:       C1 B21C Courses not offered by school available at other district high school
Instrument:  counselor_BY
Description: In which of the following ways may a student take a course for credit if it is not offered by your school?
(Check all that apply.)
  Independent study
  On-line or distance learning courses
  Courses at another traditional high school in the district
  Courses at a local career or technical school
  Courses at a 

In [39]:
assignments = json.load(open("exclusions.json"))

for variable, decision in assignments.items():
    if decision["certain"]:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    print(f"Current:  exclude={decision['exclude']}")
    
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    exclude = input("\nExclude? (y/n/q to quit): ").strip().lower()
    if exclude == "q":
        break
    
    assignments[variable]["exclude"] = (exclude == "y")
    assignments[variable]["certain"] = True
    
    json.dump(assignments, open("exclusions.json", "w"), indent=2)

print("Done")


Variable: C1GOAL3
Current:  exclude=False
Label:    C1 A08 School counseling program's third most emphasized goal
Description: Of the two goals remaining, which one does your school's counseling program emphasize more? Would you say...
  helping students plan and prepare for their work roles after high school
  helping students with personal growth and development
  helping students plan and prepare for postsecondary school


Done


In [36]:
assignments = json.load(open("exclusions.json"))

excluded = {k: v for k, v in assignments.items() if v['exclude']}
print(f"Currently excluded: {len(excluded)}")
print()

for variable, decision in excluded.items():
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    keep_excluded = input("\nKeep excluded? (y/n/q to quit): ").strip().lower()
    if keep_excluded == "q":
        break
    
    if keep_excluded == "n":
        assignments[variable]["exclude"] = False
    
    assignments[variable]["certain"] = True
    json.dump(assignments, open("exclusions.json", "w"), indent=2)

print("Done")

Currently excluded: 52


Variable: X1PARRESP
Label:    X1 Whether parent questionnaire respondent is Parent 1
Description: Indicates whether or not the parent questionnaire respondent is "parent #1"; that is, the parent to whom all "parent #1" variables (e.g. X1P1RELATION, X1PAR1EMP, P1YRBORN1, P1USYR1, etc.) refer.  The parent questionnaire respondent is always "parent #1" except in cases where: (1) the respondent is a

Variable: X1MOMRESP
Label:    X1 Whether parent questionnaire respondent is mother
Description: Indicates whether or not the parent questionnaire respondent is a biological, adoptive, or step mother.  X1MOMRESP is derived from three composite variables (X1P1RELATION, X1P2RELATION, and X1PARRESP).


                                                                        Frequency             Pe

Variable: X1DADRESP
Label:    X1 Whether parent questionnaire respondent is father
Description: Indicates whether or not the parent questionnaire respondent is a biological, ado

In [47]:
tier1 = json.load(open("tier1_assignments.json"))
tier2 = json.load(open("tier2_assignments.json"))
exclusions = json.load(open("exclusions.json"))

already_assigned = set()
for k, v in tier1.items():
    if v["tier_1"]:
        already_assigned.add(k)
for k, v in tier2.items():
    if v["tier_2"]:
        already_assigned.add(k)
for k, v in exclusions.items():
    if v["exclude"]:
        already_assigned.add(k)

SAVE_FILE = "tier3_assignments.json"

if os.path.exists(SAVE_FILE):
    assignments = json.load(open(SAVE_FILE))
else:
    assignments = {}

last_variable = None

for _, row in codebook_df.iterrows():
    v = row["variable"]

    if row["instrument"] in ("identifier", "target"):
        continue
    if v in already_assigned:
        continue
    if v in assignments:
        continue

    print(f"\n{'='*60}")
    print(f"Variable:    {v}")
    print(f"Label:       {row['label']}")
    print(f"Instrument:  {row['instrument']}")
    print(f"Description: {row['description'][:400]}")

    tier3 = input("\nTier 3? (y/n/r to redo last/q to quit): ").strip().lower()
    if tier3 == "q":
        break
    elif tier3 == "r":
        if last_variable:
            assignments[last_variable]["certain"] = False
            json.dump(assignments, open(SAVE_FILE, "w"), indent=2)
            print(f"Marked {last_variable} as uncertain — will reappear next run")
        else:
            print("Nothing to redo")
        continue

    certain = input("Certain? (y/n): ").strip().lower()

    assignments[v] = {
        "tier_3": tier3 == "y",
        "certain": certain == "y",
    }
    last_variable = v

    json.dump(assignments, open(SAVE_FILE, "w"), indent=2)

remaining = 911 - len(already_assigned) - len(assignments)
print(f"\nProgress: {len(assignments)} Tier 3 decisions made, ~{remaining} remaining")


Variable:    C1HSTOWORKNO
Label:       C1 B32T School doesn't assist students with transition from high school to work
Instrument:  counselor_BY
Description: In which of the following ways does the school assist students with the transition from high school to work?
(Check all that apply.)
  Internships with local employers
  Job fairs
  Career guides or skills assessments such as KUDER, HIRE, What Color is Your Parachute
  School or classroom presentations by local employers
  Career awareness activities
  School courses in career decision making
  Ca

Variable:    C1G9MMSCNSL
Label:       C1 C02A Importance of MS counselor recommendation for 9th grade math placement
Instrument:  counselor_BY
Description: How important is each of the following factors in placing a typical 9th grade student into a mathematics course?
Middle school counselor recommendation
  Not at all important
  A little important
  Somewhat important
  Very important


                                               

In [43]:
assignments = json.load(open("tier3_assignments.json"))

for variable, decision in assignments.items():
    if decision["certain"]:
        continue
    
    print(f"\n{'='*60}")
    print(f"Variable: {variable}")
    print(f"Current:  tier_3={decision['tier_3']}")
    
    row = codebook_df[codebook_df["variable"] == variable]
    if not row.empty:
        print(f"Label:    {row.iloc[0]['label']}")
        print(f"Description: {row.iloc[0]['description'][:300]}")
    
    tier3 = input("\nTier 3? (y/n/q to quit): ").strip().lower()
    if tier3 == "q":
        break
    
    assignments[variable]["tier_3"] = (tier3 == "y")
    assignments[variable]["certain"] = True
    
    json.dump(assignments, open("tier3_assignments.json", "w"), indent=2)

print("Done")


Variable: S1NOMSACT
Current:  tier_3=True
Label:    S1 B04I 9th grader did not participate in any math/science activities listed
Description: Since the beginning of the last school year (2008-2009), which of the following activities have you participated in?
(Check all that apply.)
  Math club
  Math competition
  Math camp
  Math study groups or a program where you were tutored in math
  Science club
  Science competition
  Science camp


Variable: S1MUNDERST
Current:  tier_3=True
Label:    S1 C02 How often 9th grader thinks he/she really understands math assignments
Description: When you are working on a math assignment, how often do you think you really understand the assignment?
  Never
  Rarely
  Sometimes
  Often


                                                                        Frequency             Percent
Done


In [49]:
tier1 = json.load(open("tier1_assignments.json"))
tier2 = json.load(open("tier2_assignments.json"))
exclusions = json.load(open("exclusions.json"))
tier3 = json.load(open("tier3_assignments.json"))

col_names = open("hsls_column_names.txt").read().strip().split("\n")

tier1_vars = {k for k, v in tier1.items() if v["tier_1"]}
tier2_vars = {k for k, v in tier2.items() if v["tier_2"]}
excluded_vars = {k for k, v in exclusions.items() if v["exclude"]}
tier3_vars = {k for k, v in tier3.items() if v["tier_3"]}

assigned = tier1_vars | tier2_vars | excluded_vars | tier3_vars

tier4_vars = [c for c in col_names if c not in assigned
              and c != "STU_ID" and c != "X4EVERDROP"]

tier4_assignments = {v: {"tier_4": True, "certain": True} for v in tier4_vars}

json.dump(tier4_assignments, open("tier4_assignments.json", "w"), indent=2)
print(f"Tier 4 assignments saved: {len(tier4_assignments)} variables")

Tier 4 assignments saved: 397 variables


In [50]:
tier1 = json.load(open("tier1_assignments.json"))
tier2 = json.load(open("tier2_assignments.json"))
tier3 = json.load(open("tier3_assignments.json"))
tier4 = json.load(open("tier4_assignments.json"))
exclusions = json.load(open("exclusions.json"))

col_names = open("hsls_column_names.txt").read().strip().split("\n")

# Build master assignment for every variable
master = {}
for v in col_names:
    if v in ("STU_ID", "X4EVERDROP"):
        continue
    
    if v in tier1 and tier1[v]["tier_1"]:
        master[v] = 1
    elif v in tier2 and tier2[v]["tier_2"]:
        master[v] = 2
    elif v in exclusions and exclusions[v]["exclude"]:
        master[v] = None  # excluded
    elif v in tier3 and tier3[v]["tier_3"]:
        master[v] = 3
    else:
        master[v] = 4

# Build cumulative feature lists
tier1_features = [v for v, t in master.items() if t == 1]
tier2_features = [v for v, t in master.items() if t in (1, 2)]
tier3_features = [v for v, t in master.items() if t in (1, 2, 3)]
tier4_features = [v for v, t in master.items() if t in (1, 2, 3, 4)]

print(f"Tier 1 features: {len(tier1_features)}")
print(f"Tier 2 features: {len(tier2_features)}")
print(f"Tier 3 features: {len(tier3_features)}")
print(f"Tier 4 features: {len(tier4_features)}")
print(f"Excluded: {sum(1 for t in master.values() if t is None)}")

# Save as JSON
output = {
    "tier1": tier1_features,
    "tier2": tier2_features,
    "tier3": tier3_features,
    "tier4": tier4_features,
}

json.dump(output, open("feature_tiers.json", "w"), indent=2)
print("\nSaved to feature_tiers.json")

Tier 1 features: 130
Tier 2 features: 192
Tier 3 features: 436
Tier 4 features: 833
Excluded: 78

Saved to feature_tiers.json
